# Training and evaluating DistilBERT on headline level classification

In [2]:
import json
import pandas as pd
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

colab = False

if colab == True:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/data")
    models_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/models/DistilBERT_headline_runs")
else:
    data_path = "../data"
    models_path = "../models/DistilBERT_headline_runs"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

W1014 16:09:14.302000 23024 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


### Transform data into appropriate format

Create Python list of dicts

In [3]:
def flatten_transform(target_set):

    modified_set = []

    for item in target_set:
        date = item["date"]
        answer = item["answer"]
        for index, headline in enumerate(item["headlines"]):
            if index == answer:
                fake = 1
            else:
                fake = 0
            entry = {
                "Date": date,
                "Headline": headline,
                "Fake": fake
            }
            modified_set.append(entry)

    return modified_set

train_set_nn_ready = flatten_transform(train_nn_set)
validate_set_nn_ready = flatten_transform(validate_set)
test_set_nn_ready = flatten_transform(test_set)

Transform into dataframe and create X and y classes

In [4]:
train_nn_df = pd.DataFrame(train_set_nn_ready)
X_train_nn_text = train_nn_df["Headline"]
y_train = train_nn_df["Fake"]

validate_nn_df = pd.DataFrame(validate_set_nn_ready)
X_validate_nn_text = validate_nn_df["Headline"]
y_validate = validate_nn_df["Fake"]

test_df = pd.DataFrame(test_set_nn_ready)
X_test_nn_text = test_df["Headline"]
y_test = test_df["Fake"]

### DistilBERT

Initial setup and tokenisation

In [5]:
# Set the seed
seed = 42
set_seed(seed)

# Choose model
model_name = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define tokenizer settings (need to think about this)
truncation = True
padding = "max_length"
max_length = 50

# Tokenize dataset
train_encodings = tokenizer(
    list(X_train_nn_text),
    truncation=truncation,
    padding=padding,
    max_length=max_length
    )
validate_encodings = tokenizer(
    list(X_validate_nn_text),
    truncation=truncation,
    padding=padding,
    max_length=max_length
    )
test_encodings = tokenizer(
    list(X_test_nn_text),
    truncation=truncation,
    padding=padding,
    max_length=max_length
    )

# Convert to torch Dataset
class HeadlineDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Set up train, validation and test datasets
train_dataset = HeadlineDataset(train_encodings, list(y_train))
validate_dataset = HeadlineDataset(validate_encodings, list(y_validate))
test_dataset = HeadlineDataset(test_encodings, list(y_test))

# Size of train and test datasets
n_train = len(train_dataset)
n_validate = len(validate_dataset)
n_test = len(test_dataset)

In [15]:
token_lengths = [len(tokenizer.encode(text)) for text in X_validate_nn_text]
print(max(token_lengths))
print(np.percentile(token_lengths, [50, 90, 95, 99]))

47
[17.   24.   26.   32.99]


Model hyperparameters and training (only run in Colab)

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
num_model_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Define metadata saving functions
def create_run_dir(base=models_path):
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_dir = Path(base) / timestamp
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

# Create run directories
run_dir = create_run_dir()

# Define training args
output_dir = str(run_dir / "checkpoints")
num_train_epochs=20
per_device_train_batch_size=32
per_device_eval_batch_size=268
eval_strategy="epoch"
eval_steps=1
save_strategy="epoch"
save_steps=1
disable_tqdm=True
logging_steps=1
report_to="none"
learning_rate=5e-5 # Default is 5e-5
optim="adamw_torch" # Default is "adamw_torch"
weight_decay=0 # Default is 0
warmup_steps=0 # Default is 0
lr_scheduler_type="linear" # Default is "linear"
load_best_model_at_end=True
metric_for_best_model="eval_loss"
greater_is_better=False
early_stopping_patience=3

# Assign training args
training_args = TrainingArguments(
    seed=seed,
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    eval_strategy=eval_strategy,
    eval_steps=eval_steps,
    save_strategy=save_strategy,
    disable_tqdm=disable_tqdm,
    logging_steps=logging_steps,
    report_to=report_to,
    learning_rate=learning_rate,
    optim=optim,
    weight_decay=weight_decay,
    warmup_steps=warmup_steps,
    lr_scheduler_type=lr_scheduler_type,
    load_best_model_at_end=load_best_model_at_end,
    metric_for_best_model=metric_for_best_model,
    greater_is_better=greater_is_better
)

# Early stopping option
callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)] # If "None", comment line in Trainer

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validate_dataset,
    callbacks=callbacks
)

# Train
output = trainer.train()

# Best validation loss and the step when it happened
best_step = trainer.state.best_step if hasattr(trainer.state, "best_step") else None
best_metric = trainer.state.best_metric if hasattr(trainer.state, "best_metric") else None

# Tiered run metadata
run_metadata = {
    "Model Information": {
        "Model name": model_name,
        "Number of model parameters": num_model_params,
        "Number of trainable parameters": trainable_params,
    },
    "Dataset Information": {
        "Training set size": n_train,
        "Validation set size": n_validate,
        "Test set size": n_test,
    },
    "Tokenizer Settings": {
        "Truncation setting": truncation,
        "Padding setting": padding,
        "Max length": max_length,
    },
    "Training Hyperparameters": {
        "Number of training epochs": num_train_epochs,
        "Per-device training batch size": per_device_train_batch_size,
        "Per-device eval batch size": per_device_eval_batch_size,
        "Number of evaluation steps": eval_steps,
        "Output directory": output_dir,
        "Save strategy": save_strategy,
        "Save steps": save_steps,
        "Disable tqdm": disable_tqdm,
        "Logging steps": logging_steps,
        "Report to": report_to,
        "Learning rate": learning_rate,
        "Optimiser": optim,
        "Weight decay": weight_decay,
        "Warmup steps": warmup_steps,
        "Learning rate scheduler type": lr_scheduler_type,
        "Callbacks": [type(cb).__name__ for cb in callbacks],
        "Seed": seed,
        "Load best model at end": load_best_model_at_end,
        "Metric for best model": metric_for_best_model,
        "Greater is better": greater_is_better,
        "Early stopping patience": early_stopping_patience,
    },
    "Training Results": {
        "Training time and speed": output.metrics,
        "Best model checkpoint": trainer.state.best_model_checkpoint,
        "Best validation loss": trainer.state.best_metric,
    }
}

def convert_numpy(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (np.floating, float)):
        return round(float(obj), 6)
    else:
        return obj

with open(f"{run_dir}/run_metadata.json", "w") as f:
    json.dump(convert_numpy(run_metadata), f, indent=4)

print(f"Run metadata saved to {run_dir}/run_metadata.json")

# Extract log history
log_history = trainer.state.log_history

# Training loss
train_points = [x for x in log_history if "loss" in x]
train_steps = [p["step"] for p in train_points]
train_losses = [p["loss"] for p in train_points]

# Validation loss
val_points = [x for x in log_history if "eval_loss" in x]
val_steps = [p["step"] for p in val_points]
val_losses = [p["eval_loss"] for p in val_points]

# Plot
plt.figure(figsize=(8,5))
plt.plot(train_steps, train_losses, label="Training Loss")
plt.plot(val_steps, val_losses, label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(run_dir / "loss_curve.png")
plt.close()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.6933', 'grad_norm': '2.217', 'learning_rate': '5e-05', 'epoch': '0.03846'}
{'loss': '0.6341', 'grad_norm': '1.821', 'learning_rate': '4.99e-05', 'epoch': '0.07692'}
{'loss': '0.5896', 'grad_norm': '1.666', 'learning_rate': '4.981e-05', 'epoch': '0.1154'}
{'loss': '0.6651', 'grad_norm': '1.23', 'learning_rate': '4.971e-05', 'epoch': '0.1538'}
{'loss': '0.4917', 'grad_norm': '1.275', 'learning_rate': '4.962e-05', 'epoch': '0.1923'}
{'loss': '0.4526', 'grad_norm': '1.319', 'learning_rate': '4.952e-05', 'epoch': '0.2308'}
{'loss': '0.5516', 'grad_norm': '1.124', 'learning_rate': '4.942e-05', 'epoch': '0.2692'}
{'loss': '0.3847', 'grad_norm': '1.081', 'learning_rate': '4.933e-05', 'epoch': '0.3077'}
{'loss': '0.6062', 'grad_norm': '1.985', 'learning_rate': '4.923e-05', 'epoch': '0.3462'}
{'loss': '0.4938', 'grad_norm': '1.076', 'learning_rate': '4.913e-05', 'epoch': '0.3846'}
{'loss': '0.4673', 'grad_norm': '1.235', 'learning_rate': '4.904e-05', 'epoch': '0.4231'}
{'loss': '0.94

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.5304', 'grad_norm': '2.162', 'learning_rate': '4.75e-05', 'epoch': '1.038'}
{'loss': '0.5056', 'grad_norm': '1.561', 'learning_rate': '4.74e-05', 'epoch': '1.077'}
{'loss': '0.4726', 'grad_norm': '1.936', 'learning_rate': '4.731e-05', 'epoch': '1.115'}
{'loss': '0.5026', 'grad_norm': '2.608', 'learning_rate': '4.721e-05', 'epoch': '1.154'}
{'loss': '0.4967', 'grad_norm': '1.988', 'learning_rate': '4.712e-05', 'epoch': '1.192'}
{'loss': '0.5068', 'grad_norm': '1.574', 'learning_rate': '4.702e-05', 'epoch': '1.231'}
{'loss': '0.4291', 'grad_norm': '2.574', 'learning_rate': '4.692e-05', 'epoch': '1.269'}
{'loss': '0.5319', 'grad_norm': '1.859', 'learning_rate': '4.683e-05', 'epoch': '1.308'}
{'loss': '0.5012', 'grad_norm': '1.813', 'learning_rate': '4.673e-05', 'epoch': '1.346'}
{'loss': '0.4202', 'grad_norm': '2.636', 'learning_rate': '4.663e-05', 'epoch': '1.385'}
{'loss': '0.3597', 'grad_norm': '3.227', 'learning_rate': '4.654e-05', 'epoch': '1.423'}
{'loss': '0.4132', 'gra

...

{'loss': '0.3292', 'grad_norm': '5.141', 'learning_rate': '4.5e-05', 'epoch': '2.038'}
{'loss': '0.2914', 'grad_norm': '5.658', 'learning_rate': '4.49e-05', 'epoch': '2.077'}
{'loss': '0.3244', 'grad_norm': '2.566', 'learning_rate': '4.481e-05', 'epoch': '2.115'}
{'loss': '0.2684', 'grad_norm': '4.139', 'learning_rate': '4.471e-05', 'epoch': '2.154'}
{'loss': '0.2381', 'grad_norm': '3.475', 'learning_rate': '4.462e-05', 'epoch': '2.192'}
{'loss': '0.324', 'grad_norm': '3.199', 'learning_rate': '4.452e-05', 'epoch': '2.231'}
{'loss': '0.1929', 'grad_norm': '2.182', 'learning_rate': '4.442e-05', 'epoch': '2.269'}
{'loss': '0.1752', 'grad_norm': '3.149', 'learning_rate': '4.433e-05', 'epoch': '2.308'}
{'loss': '0.2678', 'grad_norm': '5.66', 'learning_rate': '4.423e-05', 'epoch': '2.346'}
{'loss': '0.2012', 'grad_norm': '1.999', 'learning_rate': '4.413e-05', 'epoch': '2.385'}
{'loss': '0.1334', 'grad_norm': '1.846', 'learning_rate': '4.404e-05', 'epoch': '2.423'}
{'loss': '0.1507', 'grad_n

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.1517', 'grad_norm': '3.285', 'learning_rate': '4.25e-05', 'epoch': '3.038'}
{'loss': '0.2052', 'grad_norm': '3.023', 'learning_rate': '4.24e-05', 'epoch': '3.077'}
{'loss': '0.06737', 'grad_norm': '2.57', 'learning_rate': '4.231e-05', 'epoch': '3.115'}
{'loss': '0.03968', 'grad_norm': '1.114', 'learning_rate': '4.221e-05', 'epoch': '3.154'}
{'loss': '0.04358', 'grad_norm': '2.41', 'learning_rate': '4.212e-05', 'epoch': '3.192'}
{'loss': '0.126', 'grad_norm': '3.061', 'learning_rate': '4.202e-05', 'epoch': '3.231'}
{'loss': '0.02969', 'grad_norm': '0.9252', 'learning_rate': '4.192e-05', 'epoch': '3.269'}
{'loss': '0.04031', 'grad_norm': '2.017', 'learning_rate': '4.183e-05', 'epoch': '3.308'}
{'loss': '0.03817', 'grad_norm': '1.192', 'learning_rate': '4.173e-05', 'epoch': '3.346'}
{'loss': '0.07628', 'grad_norm': '7.403', 'learning_rate': '4.163e-05', 'epoch': '3.385'}
{'loss': '0.04349', 'grad_norm': '2.244', 'learning_rate': '4.154e-05', 'epoch': '3.423'}
{'loss': '0.2235'

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '56.16', 'train_samples_per_second': '289.2', 'train_steps_per_second': '9.259', 'train_loss': '0.3335', 'epoch': '4'}
Run metadata saved to /content/drive/MyDrive/Colab Notebooks/3H1L/models/DistilBERT_headline_runs/20261014-150153/run_metadata.json
[0.6933091878890991, 0.6340833306312561, 0.589556097984314, 0.6651482582092285, 0.49174338579177856]


Logging and storing (only run in Colab)

In [42]:
log_history = trainer.state.log_history

clean_log = []
last_train_loss = None
last_step = None

# Record values at each logging step
for entry in log_history:
    step = entry.get("step")
    if "loss" in entry:
        last_train_loss = entry["loss"]
    if "eval_loss" in entry:
        # Skip if this step is already appended
        if step == last_step:
            continue
        clean_entry = {
            "step": step,
            "train_loss": last_train_loss,
            "eval_loss": entry.get("eval_loss"),
            "epoch": entry.get("epoch")
        }
        clean_log.append(clean_entry)
        last_step = step

# Save JSON
with open(run_dir / "log_history.json", "w") as f:
    json.dump(clean_log, f, indent=2)

# Save CSV
df = pd.DataFrame(clean_log)
df.to_csv(run_dir / "log_history.csv", index=False)

# Save best model
best_model_dir = run_dir / "best_model"
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/3H1L/models/DistilBERT_headline_runs/20261014-150153/best_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/3H1L/models/DistilBERT_headline_runs/20261014-150153/best_model/tokenizer.json')

Load model for evaluation

In [6]:
model_path = Path(f"{models_path}/20261014-150153/best_model")
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

test_inputs = {
    key: torch.tensor(value)
    for key, value in test_encodings.items()
}

with torch.no_grad():
    outputs = model(**test_inputs)

logits = outputs.logits

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Evaluate

In [7]:
# Calculate the probability that each headline is fake
test_df["Fake Probability DistilBERT"] = (
    torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
)

# Group headlines by quiz
test_by_quiz_df = test_df.groupby("Date")

correct = 0

for date, quiz_df in test_by_quiz_df:
    # Headline with the highest predicted fake probability
    predicted_index = quiz_df["Fake Probability DistilBERT"].idxmax()

    # Actual fake headline
    actual_index = quiz_df["Fake"].idxmax()

    if predicted_index == actual_index:
        correct += 1

accuracy_DistilBERT_headline = correct / len(test_set)

print(f"Quiz-level accuracy: {accuracy_DistilBERT_headline:.2%}")

Quiz-level accuracy: 46.27%


Save results

In [8]:
# Calculate probabilites using softmax
fake_probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

# Arrange probabilites by quiz
quiz_fake_probs = fake_probs.reshape(-1, 4)

# Take the prediction as the highest probability per quiz
predictions = np.argmax(quiz_fake_probs, axis=1)

# Take confidence as the probability value for each prediction
confidence = quiz_fake_probs[
    np.arange(len(predictions)),
    predictions
]

# Dates from the fixed test set
test_dates = [quiz["date"] for quiz in test_set]

# Create prediction dataframe
predictions_DistilBERT = pd.DataFrame({
    "date": test_dates,
    "prediction": predictions,
    "confidence": confidence
})

predictions_DistilBERT.to_csv(
    "../data/predictions_distilbert_headline.csv",
    index=False
)

Checking changes from old to new models

In [10]:
old = pd.read_csv(f"{data_path}/old/predictions_distilbert_headline.csv")
new = pd.read_csv(f"{data_path}/predictions_distilbert_headline.csv")

m = old[["date", "prediction"]].merge(
    new[["date", "prediction"]], on="date", how="outer",
    suffixes=("_old", "_new"), indicator=True
)

print(m["_merge"].value_counts())  # dates present in only one file
diff = m[(m["_merge"] == "both") & (m["prediction_old"] != m["prediction_new"])]
print(f"{len(diff)} of {len(m)} predictions changed")
print(diff.drop(columns="_merge").to_string(index=False))

_merge
both          67
left_only      0
right_only     0
Name: count, dtype: int64
1 of 67 predictions changed
      date  prediction_old  prediction_new
2022-01-27               1               3
